**Robustness under Long and Concurrent Sessions.** The experiment bundle separates steady-state tail latency from recovery correctness: (a-b) p50/p95 costs over a 64-call trajectory, (c) four concurrently isolated agents, and (d) explicit worker-crash, session-resume, and cross-agent isolation invariants. Heterogeneous scenarios are not collapsed into one latency score.

In [ ]:
# ipython -c "%run plot_robustness.ipynb"

import json
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

STANDARD_WIDTH = 17.8

def cm_to_inch(value):
    return value / 2.54

plt.rcParams.update(plt.rcParamsDefault)
matplotlib.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'Nimbus Roman'
plt.rcParams['axes.grid'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['legend.frameon'] = True
plt.rcParams['legend.edgecolor'] = '0.55'
plt.rcParams['legend.framealpha'] = 1.0
plt.rcParams['legend.fancybox'] = False

P50 = dict(color='#4f9fcf', marker='^', linestyle='-.', linewidth=0.9, markersize=3.2, markerfacecolor='none')
P95 = dict(color='#c00000', marker='s', linestyle='-', linewidth=1.0, markersize=3.2)
AGENT = dict(color='black', marker='o', linestyle='--', linewidth=0.8, markersize=2.8, markerfacecolor='none')

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'motivation' else cwd
RESULTS = ROOT / 'experiments' / 'results'
FIGDIR = ROOT / 'motivation'
records = json.loads((RESULTS / 'robustness.json').read_text())
tail = pd.DataFrame(item for item in records if item['suite'] == 'p50_p95')
tail['label'] = tail['mode'].map({
    'agenttx_without_read_tracing': 'no-trace',
    'agenttx_full': 'full',
})
worker = next(item for item in records if item['suite'] == 'worker_crash')
session = next(item for item in records if item['suite'] == 'long_session')
concurrent = next(item for item in records if item['suite'] == 'concurrent_agents')

fig = plt.figure(dpi=300, figsize=(cm_to_inch(STANDARD_WIDTH), cm_to_inch(7.0)))
x = range(len(tail))

ax = plt.subplot(2, 2, 1)
step50, = ax.plot(x, tail['step_p50_ms'], **P50, label='p50')
step95, = ax.plot(x, tail['step_p95_ms'], **P95, label='p95')
ax.set_ylabel('Per-call latency (ms)', fontsize=8)
ax.set_xlabel('AgentTX mode\n(a) Call tail', fontsize=7)
ax.set_xticks(list(x), tail['label'])

ax = plt.subplot(2, 2, 2)
run50, = ax.plot(x, tail['run_p50_ms'] / 1000.0, **P50, label='p50')
run95, = ax.plot(x, tail['run_p95_ms'] / 1000.0, **P95, label='p95')
ax.set_ylabel('Trajectory latency (s)', fontsize=8)
ax.set_xlabel('AgentTX mode\n(b) 64-call workload', fontsize=7)
ax.set_xticks(list(x), tail['label'])

ax = plt.subplot(2, 2, 3)
details = pd.DataFrame(concurrent['details']).sort_values('agent_id')
agents, = ax.plot(details['agent_id'] + 1, details['wall_ms'] / 1000.0, **AGENT, label='per-agent latency')
ax.axhline(concurrent['wall_ms'] / 1000.0, color='#c00000', linestyle=':', linewidth=0.8, label='joint wall time')
ax.set_ylabel('Latency (s)', fontsize=8)
ax.set_xlabel('Concurrent agent\n(c) Isolated execution', fontsize=7)
ax.set_xticks(details['agent_id'] + 1)

ax = plt.subplot(2, 2, 4)
checks = [
    ('Fallback', worker['fallback_used']),
    ('Restart', worker['worker_restarted']),
    ('Resume', session['resumed']),
    ('256 files', session['materialized_files'] == session['steps']),
    ('4/4 agents', concurrent['successful_agents'] == concurrent['agents']),
    ('No mixing', not concurrent['cross_contamination']),
]
status, = ax.plot(range(len(checks)), [100.0 if passed else 0.0 for _, passed in checks], **P95, label='invariant satisfied')
ax.set_ylabel('Invariant satisfied (%)', fontsize=8)
ax.set_xlabel('Injected scenario/property\n(d) Recovery and isolation', fontsize=7)
ax.set_xticks(range(len(checks)), [label for label, _ in checks], rotation=20, ha='right')
ax.set_ylim(-5, 105)

for ax in fig.axes:
    ax.tick_params(axis='both', labelsize=7)

fig.legend(handles=[step50, step95, agents, status],
           loc='upper center', bbox_to_anchor=(0.5, 1.035), ncol=4,
           fontsize=6.7, columnspacing=0.9, handlelength=1.7, handletextpad=0.35, borderpad=0.3)
plt.tight_layout(pad=0.6, h_pad=1.6, w_pad=1.2, rect=[0.0, 0.0, 1.0, 0.91])
plt.savefig(FIGDIR / 'FIG-Robustness.pdf', bbox_inches='tight', pad_inches=0.02,
            metadata={'CreationDate': None, 'ModDate': None})
plt.savefig(FIGDIR / 'FIG-Robustness.png', dpi=300, bbox_inches='tight', pad_inches=0.02)
plt.show()

print(f"worker fallback={worker['fallback_ms']:.1f} ms; long session={session['steps']} steps; concurrent={concurrent['successful_agents']}/{concurrent['agents']} isolated agents")
